# Behavior Classification: CNN + LSTM

Notebook ini dikonfigurasi untuk dijalankan di **Antigravity (Local)** maupun **Google Colab (Cloud)**.

---

### Khusus Google Colab (Cloud)
Jika Anda menjalankan ini di Colab, jalankan sel di bawah ini untuk meng-clone project dari GitHub.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    # 1. Clone Repositori
    !git clone https://github.com/mdrnid/skeleton-based-focus-estimation.git
    
    # 2. Masuk ke folder project
    %cd skeleton-based-focus-estimation
    
    print("\u2705 Project berhasil dicolone dan siap digunakan.")

In [ ]:
import os
import sys
from pathlib import Path

# 1. DETEKSI PATH & LINGKUNGAN YANG LEBIH CERDAS
def setup_paths():
    # Cek apakah kita ada di dalam folder hasil clone
    curr_dir = Path(os.getcwd())
    
    # Urutan pengecekan: 
    # 1. Folder saat ini (jika kita sudah %cd ke repo)
    # 2. Folder satu tingkat di atas (lokal Antigravity notebooks/)
    potential_roots = [curr_dir, curr_dir.parent]
    
    for root in potential_roots:
        check_path = root / "data" / "processed"
        if check_path.exists():
            print(f"\u2705 Root Proyek Ditemukan: {root.absolute()}")
            return str(root.absolute())

    # Fallback terakhir jika semua gagal
    return '..'

BASE_PATH = setup_paths()
DATA_PATH = os.path.join(BASE_PATH, 'data', 'processed')
MODEL_PATH = os.path.join(BASE_PATH, 'models')

os.makedirs(MODEL_PATH, exist_ok=True)
print(f"Final Project Root: {BASE_PATH}")
print(f"Final Data Path: {DATA_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

## 1. Load Data

In [ ]:
x_file = os.path.join(DATA_PATH, 'X.npy')
y_file = os.path.join(DATA_PATH, 'y.npy')

if os.path.exists(x_file):
    X = np.load(x_file)
    y = np.load(y_file)
    print(f"X shape: {X.shape} (Sampel, Timesteps, Fitur)")
    print(f"y shape: {y.shape}")
else:
    print(f"\u274c ERROR: File data tidak ditemukan di {DATA_PATH}")

## 2. Data Splitting (70/15/15)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

## 3. Training CNN + LSTM

In [ ]:
model = Sequential([
    Input(shape=(10, 68)),
    Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Conv1D(128, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
ckpt_path = os.path.join(MODEL_PATH, 'pose_model_best.keras')
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ModelCheckpoint(ckpt_path, monitor='val_accuracy', save_best_only=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

## 4. Evaluasi

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.show()

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")
print(classification_report(y_test, y_pred, target_names=['Tidak Fokus', 'Fokus']))